# Solutions — Data Validation: Required, Types, Rules, References, Schemas

Exercise notebook: `08_data_engineering_python/02_data_validation.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Required fields

**Working solution**

In [ ]:
def missing_fields(record, required):
    def blank(v):
        return v is None or (isinstance(v, str) and not v.strip())
    return [f for f in required if f not in record or blank(record[f])]

**Explanation**

Truthiness is wrong for data validation: `0` and `False` are legitimate values. Write the exact rule.

**Complexity:** O(fields).

**Common mistakes**

- `if not record.get(f)` → rejects quantity 0 and is_active False.

## Exercise 2 — 🟡 Medium · Type checks against string data

**Working solution**

In [ ]:
from datetime import date, datetime

def _is_bool(v):
    return v.strip().lower() in {"true", "false", "1", "0", "yes", "no"}

def _parses(fn):
    def check(v):
        try:
            fn(v.strip())
            return True
        except ValueError:
            return False
    return check

CHECKS = {
    "int": _parses(int),
    "float": _parses(float),
    "bool": _is_bool,
    "date": _parses(date.fromisoformat),
    "datetime": _parses(lambda s: datetime.fromisoformat(s.replace("Z", "+00:00"))),
    "str": lambda v: True,
}

def type_errors(record, schema):
    unknown = set(schema.values()) - CHECKS.keys()
    if unknown:
        raise ValueError(f"unknown types: {sorted(unknown)}")
    errors = {}
    for field, type_name in schema.items():
        value = record.get(field)
        if value is None or (isinstance(value, str) and not value.strip()):
            continue
        if not CHECKS[type_name](str(value)):
            errors[field] = f"expected {type_name}, got {value!r}"
    return errors

**Explanation**

A dict from type name → checker function is a dispatch table: adding a new type is one line. `_parses(fn)` is a small higher-order function turning 'raises on bad input' into 'returns bool'.

**Complexity:** O(fields).

**Common mistakes**

- `date.fromisoformat` in 3.11+ accepts more formats (`20260105`) — decide if that's OK for your data contract.

## Exercise 3 — 🟡 Medium · A tiny rule engine

**Working solution**

In [ ]:
def evaluate_rules(records, rules, id_field):
    results = {name: {"failed": 0, "failed_ids": []} for name in rules}
    total = 0
    for record in records:
        total += 1
        for name, rule in rules.items():
            try:
                ok = bool(rule(record))
            except Exception:
                ok = False
            if not ok:
                res = results[name]
                res["failed"] += 1
                if len(res["failed_ids"]) < 5:
                    res["failed_ids"].append(record.get(id_field))
    for res in results.values():
        res["pass_rate"] = round(1 - res["failed"] / total, 4) if total else 1.0
    return {"total": total, "rules": results}

**Explanation**

Rules as named lambdas keep data-quality checks declarative and easy to extend. Sampling a few failing ids is what makes a DQ report actionable.

**Complexity:** O(n × rules).

**Common mistakes**

- Letting a `TypeError` inside a rule abort the whole report.

## Exercise 4 — 🟠 Hard · Referential and cross-field validation

**Working solution**

In [ ]:
def validate_orders(orders, customer_ids, catalog, max_price_deviation=0.5):
    problems = []
    for o in orders:
        errors = []
        if o["customer_id"] not in customer_ids:
            errors.append(f"unknown customer {o['customer_id']}")
        catalog_price = catalog.get(o["product_id"])
        if catalog_price is None:
            errors.append(f"unknown product {o['product_id']}")
        elif catalog_price > 0:
            deviation = abs(o["unit_price"] - catalog_price) / catalog_price
            if deviation > max_price_deviation:
                errors.append(f"price deviates {round(deviation * 100)}% from catalog")
        if o["updated_at"] < o["order_ts"]:
            errors.append("updated_at before order_ts")
        if o["status"] == "DELIVERED" and o["unit_price"] == 0:
            errors.append("delivered order with zero price")
        if errors:
            problems.append((o["order_id"], errors))
    return problems

**Explanation**

Lookups go against **sets/dicts** built once (O(1) membership). Cross-field rules (timestamps ordering, status-dependent rules) are where most real data bugs hide.

**Complexity:** O(n).

**Common mistakes**

- Checking `customer_id in list_of_customers` (O(m) each).
- Dividing by a zero catalog price.

## Challenge — 🔴 Very Hard · Declarative schema validator with a data-quality report

**Working solution**

In [ ]:
import re
from datetime import date, datetime, timezone

_TRUE, _FALSE = {"true", "1", "yes"}, {"false", "0", "no"}

def _to_bool(s):
    low = s.lower()
    if low in _TRUE:
        return True
    if low in _FALSE:
        return False
    raise ValueError(s)

def _to_dt(s):
    dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
    return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt.astimezone(timezone.utc)

CONVERTERS = {"str": str, "int": int, "float": float, "bool": _to_bool, "date": date.fromisoformat, "datetime": _to_dt}
NORMALIZERS = {"upper": str.upper, "lower": str.lower, "strip": str.strip}

class SchemaValidator:
    def __init__(self, spec):
        self.columns = spec["columns"]
        self._patterns = {c: re.compile(r["pattern"]) for c, r in self.columns.items() if r.get("pattern")}
        self._bounds = {}
        for col, rules in self.columns.items():
            conv = CONVERTERS[rules.get("type", "str")]
            self._bounds[col] = tuple(conv(rules[b]) if isinstance(rules.get(b), str) and rules.get("type") in ("date", "datetime") else rules.get(b)
                                      for b in ("min", "max"))

    def _check(self, col, raw, seen):
        rules = self.columns[col]
        value = (raw or "").strip() if isinstance(raw, str) or raw is None else raw
        if isinstance(value, str) and rules.get("normalize"):
            value = NORMALIZERS[rules["normalize"]](value)
        if value in ("", None):
            return None, ("required" if rules.get("required") else None)
        type_name = rules.get("type", "str")
        try:
            typed = CONVERTERS[type_name](value) if isinstance(value, str) else value
        except ValueError:
            return None, f"invalid {type_name}"
        if "allowed" in rules and typed not in rules["allowed"]:
            return None, "not in allowed values"
        lo, hi = self._bounds[col]
        if lo is not None and typed < lo:
            return None, "below min"
        if hi is not None and typed > hi:
            return None, "above max"
        if col in self._patterns and not self._patterns[col].fullmatch(str(typed)):
            return None, "does not match pattern"
        if rules.get("unique"):
            if typed in seen[col]:
                return None, "duplicate value"
        return typed, None

    def validate(self, rows):
        valid, rejects = [], []
        seen = {c: set() for c, r in self.columns.items() if r.get("unique")}
        stats = {c: {"nulls": 0, "errors": 0} for c in self.columns}
        total = 0
        for index, row in enumerate(rows, start=1):
            total += 1
            typed_row, errors = {}, []
            for col in self.columns:
                raw = row.get(col)
                if raw is None or (isinstance(raw, str) and not raw.strip()):
                    stats[col]["nulls"] += 1
                typed, err = self._check(col, raw, seen)
                if err:
                    errors.append(f"{col}: {err}")
                    stats[col]["errors"] += 1
                typed_row[col] = typed
            if errors:
                rejects.append({"row": index, "errors": errors, "raw": row})
                continue
            for col in seen:
                if typed_row[col] is not None:
                    seen[col].add(typed_row[col])
            valid.append(typed_row)
        report = {"rows": total, "valid": len(valid), "rejected": len(rejects), "columns": stats}
        return valid, rejects, report

**Explanation**

Configuration-driven validation separates *what* the data contract is (a dict you can keep in JSON/YAML, review in PRs, share with producers) from *how* it's enforced.
Pre-compiling regexes and pre-converting date bounds in `__init__` keeps the per-row path fast. Registering unique values only for rows that pass **all** checks avoids
a rejected row 'claiming' a key.

**Complexity:** O(rows × columns).

**Common mistakes**

- Validating uniqueness before other checks and blocking a later valid row.
- Comparing date strings with `date` objects (TypeError).